In [ ]:
import os
from google.colab import drive

In [ ]:
# 1. Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
# --- [SURGICAL AI: DATASET SANITY CHECK & VISUALIZATION] ---
import os, random, cv2
import matplotlib.pyplot as plt
from google.colab import drive

# 1. Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/DATASET_ICCSA_FINAL"
img_dir = os.path.join(base_path, "images/train")
lbl_dir = os.path.join(base_path, "labels/train")

def run_sanity_check():
    print("---  DATASET SANITY CHECK ---")
    if not os.path.exists(img_dir):
        print(" Error: Image directory not found!")
        return

    img_files = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]
    lbl_files = [f for f in os.listdir(lbl_dir) if f.endswith('.txt')]
    print(f" Found {len(img_files)} images and {len(lbl_files)} label files.")

    # Display 4 random samples to verify bounding box integrity
    print(" Generating visual samples...")
    samples = random.sample(img_files, min(4, len(img_files)))
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    classes = ["Grasper", "Hook", "Scissors", "Clipper", "Irrigator", "Bipolar", "Specimen Bag"]

    for ax, img_name in zip(axes.flatten(), samples):
        img_path = os.path.join(img_dir, img_name)
        lbl_path = os.path.join(lbl_dir, img_name.replace('.jpg', '.txt'))

        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    p = list(map(float, line.split()))
                    cid = int(p[0])
                    # Reverting YOLO format (center x, center y, width, height) back to pixels for OpenCV
                    x_center, y_center, bw, bh = p[1], p[2], p[3], p[4]
                    x1 = int((x_center - bw/2) * w)
                    y1 = int((y_center - bh/2) * h)
                    x2 = int((x_center + bw/2) * w)
                    y2 = int((y_center + bh/2) * h)

                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
                    cv2.putText(img, classes[cid], (x1, max(30, y1-10)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        ax.imshow(img)
        ax.set_title(f"Sample: {img_name}")
        ax.axis('off')

    plt.tight_layout()
    plt.show()

run_sanity_check()

In [ ]:
# --- [SURGICAL AI: DATASET SPLIT & YAML GENERATION] ---
import os, random, shutil

def prepare_yolo_workspace(base_path="/content/drive/MyDrive/DATASET_ICCSA_FINAL"):
    print(" Splitting dataset into Train (80%) and Val (20%)...")

    img_train_dir = os.path.join(base_path, "images/train")
    lbl_train_dir = os.path.join(base_path, "labels/train")
    img_val_dir = os.path.join(base_path, "images/val")
    lbl_val_dir = os.path.join(base_path, "labels/val")

    # Create validation directories if they don't exist
    os.makedirs(img_val_dir, exist_ok=True)
    os.makedirs(lbl_val_dir, exist_ok=True)

    all_imgs = [f for f in os.listdir(img_train_dir) if f.endswith('.jpg')]
    random.shuffle(all_imgs)

    # Calculate 20% index for validation set
    split_idx = int(len(all_imgs) * 0.2)
    val_imgs = all_imgs[:split_idx]

    count = 0
    for img_name in val_imgs:
        # Move image to validation folder
        shutil.move(os.path.join(img_train_dir, img_name), os.path.join(img_val_dir, img_name))

        # Move corresponding label to validation folder
        lbl_name = img_name.replace('.jpg', '.txt')
        if os.path.exists(os.path.join(lbl_train_dir, lbl_name)):
            shutil.move(os.path.join(lbl_train_dir, lbl_name), os.path.join(lbl_val_dir, lbl_name))
            count += 1

    # Generate data.yaml configuration file for YOLOv11
    yaml_content = f"""
path: {base_path}
train: images/train
val: images/val

nc: 7
names: ['Grasper', 'Hook', 'Scissors', 'Clipper', 'Irrigator', 'Bipolar', 'Specimen Bag']
"""
    with open(os.path.join(base_path, "data.yaml"), 'w') as f:
        f.write(yaml_content.strip())

    print(f" YAML Configuration Created!")
    print(f" Successfully moved {count} frames to Validation.")
    print(" Dataset is fully prepared for YOLOv11 training.")

# RUN THE SPLIT
prepare_yolo_workspace()